<a href="https://colab.research.google.com/github/MuwafagQ/Playbook-program/blob/claude%2Fsetup-gpu-video-testing-JhgUH/colab_gpu_test%20(3.1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playbook Soccer Analytics — GPU Test (Google Colab)

**Before running:** Go to `Runtime → Change runtime type` and select **T4 GPU**.

You will need:
- A **Roboflow API key** (free at roboflow.com) stored in Colab Secrets as `ROBOFLOW_API_KEY`
- A short soccer video clip (MP4, ideally 10–30 seconds for a quick test)

**Pipeline highlights (good-baseline-may9):**
- BoTSort tracker + appearance ReID (`yolo11n-cls.pt`)
- IDStabilizer — re-links IDs after occlusions using position + torso appearance
- Color-based team classification (fast, no GPU needed for this step)
- BallSmoother — interpolates missing ball detections
- HomographyStateMachine — holds last good homography through short failures
- KPI summary JSON/CSV alongside the annotated video

In [1]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU detected:', gpu.stdout.strip())
else:
    print('⚠️  No GPU found.\n'
          'Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')

GPU detected: Tesla T4, 15360 MiB


In [2]:
# ── Cell 2: System packages ───────────────────────────────────────────────────
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [3]:
# ── Cell 3: Python dependencies ───────────────────────────────────────────────
# Swap out Colab's opencv for headless (avoids display-backend conflicts)
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null

!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    onnxruntime==1.20.1 \
    tqdm \
    'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' \
    pydantic-settings==2.4.0 \
    python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' \
    'inference==1.2.2' \
    'ultralytics>=8.4.37,<8.5.0' \
    'lap>=0.5.13,<0.6'

!pip install -q 'transformers>=5.2.0,<5.3.0'

# Roboflow sports library (color-team helper, pitch config, annotators)
!pip install -q git+https://github.com/roboflow/sports.git@main

print('\n✅ All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata

In [4]:
!pip install -q pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.6 MB/s eta 0:00:00


In [5]:
# ── Cell 4: Clone the repo ────────────────────────────────────────────────────
# Public repo — no token needed. If private, replace with:
#   !git clone https://<YOUR_TOKEN>@github.com/muwafagq/playbook-program.git /content/playbook
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
!git clone --branch {BRANCH} https://github.com/muwafagq/playbook-program.git /content/playbook
!git pull origin claude/setup-gpu-video-testing-JhgUH
import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
active_branch = !git rev-parse --abbrev-ref HEAD
print('Working dir:', os.getcwd())
print('Branch:', active_branch[0])

Cloning into '/content/playbook'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 85 (delta 27), reused 33 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 5.11 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (27/27), done.
fatal: not a git repository (or any of the parent directories): .git
Working dir: /content/playbook
Branch: claude/setup-gpu-video-testing-JhgUH


In [6]:
# ── Cell 5: API key + .env ────────────────────────────────────────────────────
import shutil, os, sys
shutil.copy('baseline.env', '.env')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print('✅ Loaded API key from Colab Secrets')
except Exception:
    ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY_HERE'   # fallback
    print('⚠️  Using hardcoded API key — prefer Colab Secrets')

# Patch the API key into .env
with open('.env', 'r') as f:
    env_text = f.read()
env_text = env_text.replace('ROBOFLOW_API_KEY=', f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}')
with open('.env', 'w') as f:
    f.write(env_text)

# Export environment variables
from dotenv import dotenv_values
env_vals = dotenv_values('.env')
for k, v in env_vals.items():
    if v is not None:
        os.environ[k] = v

# FORCE CPU globally to bypass ONNX/CUDA binding issues
os.environ['DEVICE'] = 'cpu'
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY

# Patch inference library settings directly if it's already loaded
try:
    import inference.core.devices.utils as dev_utils
    dev_utils.GLOBAL_DEVICE = 'cpu'
except:
    pass

print('\n── Active model config ─────────────────────────────────')
print('PLAYER_MODEL_ID :', os.environ.get('PLAYER_MODEL_ID'))
print('FIELD_MODEL_ID  :', os.environ.get('FIELD_MODEL_ID'))
print('DEVICE          :', os.environ.get('DEVICE'))
print('────────────────────────────────────────────────────────')

✅ Loaded API key from Colab Secrets


[06/07/26 21:52:11] WARNING  Your inference package version 1.2.2 is out of date! Please upgrade to  ]8;id=7075483;file:///usr/local/lib/python3.12/dist-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=7075484;file:///usr/local/lib/python3.12/dist-packages/inference/core/__init__.py#41\41]8;;\
                             version 1.3.0 of inference for the latest features and bug fixes by                   
                             running `pip install --upgrade inference`.                                            


── Active model config ─────────────────────────────────
PLAYER_MODEL_ID : footballs-player-detection-zkams-zia6c/2
FIELD_MODEL_ID  : football-field-detection-f07vi/15
DEVICE          : cpu
────────────────────────────────────────────────────────


In [7]:
import torch, gc, sys

# 1. Clear GPU memory and cache
gc.collect()
torch.cuda.empty_cache()

# 2. Force reload the main module and its vision components to pick up CPU changes
import importlib
modules_to_reload = ['main', 'vision.detect', 'vision.utils']

for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print('✅ Modules reloaded. Ready to run on CPU.')

✅ Modules reloaded. Ready to run on CPU.


In [10]:
# @title
# ── Cell 6: Provide a test video ──────────────────────────────────────────────
# Choose ONE option and comment out the others.

# --- Option A: Upload a local file -------------------------------------------
from google.colab import files as colab_files
print('Select your MP4 file in the dialog below...')
uploaded = colab_files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print('Video ready at:', VIDEO_PATH)

# --- Option B: Download a YouTube clip (yt-dlp) ------------------------------
# !pip install -q yt-dlp
# YT_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
# !yt-dlp -o /content/test_clip.%(ext)s --recode-video mp4 -q "$YT_URL"
# VIDEO_PATH = '/content/test_clip.mp4'

# --- Option C: Mount Google Drive --------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_clip.mp4'

Select your MP4 file in the dialog below...


Saving HILAL-HAZM_match_B_up7.mp4 to HILAL-HAZM_match_B_up7.mp4
Video ready at: /content/HILAL-HAZM_match_B_up7.mp4


In [ ]:
# ── Cell 7 (optional): Trim to first N seconds ────────────────────────────────
# Skip if your clip is already short (< 30 s).
#TRIM_SECONDS = 20
#TRIMMED_PATH = '/content/playbook/test_trimmed.mp4'
#!ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c copy "{TRIMMED_PATH}" -loglevel warning
#VIDEO_PATH = TRIMMED_PATH
#print(f'Trimmed to {TRIM_SECONDS}s → {VIDEO_PATH}')

In [ ]:
# ── Cell 8: Run the pipeline (Clean Start) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, torch, gc

# Force CPU via environment variables ONLY (Safe)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['DEVICE'] = 'cpu'

# Clear any remaining memory
gc.collect()
torch.cuda.empty_cache()

# Import the pipeline
import main
import importlib
importlib.reload(main)

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
VIDEO_INPUT = '/content/playbook/HILAL-HAZM_match_B_up7.mp4'

print(f"Processing video: {VIDEO_INPUT} (Standard CPU Mode)")

try:
    main.main(
        source_video=VIDEO_INPUT,
        out_dir=OUT_DIR,
        enable_team=True,
    )
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")

Processing video: /content/playbook/HILAL-HAZM_match_B_up7.mp4 (Standard CPU Mode)
[stage] Loading models...
[stage] Models loaded.
[stage] Opening video: /content/playbook/HILAL-HAZM_match_B_up7.mp4
[stage] Video ready. total_frames=862, processing=862
WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.
[stage] Tracker mode: botsort
[stage] Color team classifier enabled (init_samples=30, margin=0.08).
[stage] Starting frame loop...


  0%|          | 0/862 [00:00<?, ?it/s]

[detect] keypoint correspondence injected: n=29 ids[0:8]=[0, 1, 2, 5, 6, 8, 9, 10]
[detect] kp class_id <-> class_name pairs: [(0, '01'), (1, '02'), (2, '03'), (5, '06'), (6, '07'), (8, '09'), (9, '10'), (10, '11'), (11, '12'), (12, '13'), (13, '15'), (14, '16')]


  0%|          | 1/862 [00:12<3:01:56, 12.68s/it]

[homography] reject reason=inlier_ratio_low (n=9 ratio=0.44 min=0.50)


...

  0%|          | 4/862 [00:47<2:46:44, 11.66s/it]

[homography] reject reason=inlier_ratio_low (n=9 ratio=0.44 min=0.50)


  1%|          | 5/862 [00:59<2:45:14, 11.57s/it]

[homography] reject reason=inlier_ratio_low (n=9 ratio=0.44 min=0.50)


  5%|▍         | 41/862 [07:53<2:39:30, 11.66s/it]

In [ ]:
# ── Cell 9: Preview annotated video ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import glob, os

# run.sh writes to outputs_test/<run_id>/; main() used OUT_DIR directly
video_file = OUT_DIR + '/annotated.mp4'
video_bytes = open(video_file, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(video_bytes).decode()
HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# ── Cell 10: KPI summary ──────────────────────────────────────────────────────
import json, pandas as pd

kpi_json = OUT_DIR + '/kpi_summary.json'
if os.path.exists(kpi_json):
    with open(kpi_json) as f:
        kpi = json.load(f)
    print(json.dumps(kpi, indent=2))
else:
    print('kpi_summary.json not found — check OUT_DIR path')

csv_path = OUT_DIR + '/per_frame_tracks.csv'
df = pd.read_csv(csv_path)
print(f'\nTracking CSV: {len(df):,} rows | {df.frame.nunique()} frames | {df.track_id.nunique()} unique IDs')
df.head(5)

In [ ]:
# ── Cell 11: Download all outputs ─────────────────────────────────────────────
from google.colab import files as colab_files
for fname in ['annotated.mp4', 'per_frame_tracks.csv', 'kpi_summary.json', 'kpi_summary.csv']:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        colab_files.download(fpath)
    else:
        print(f'Skipped (not found): {fpath}')